# UFC Fight Prediction and Betting Market Evaluation

## Core Research Question

Can historical UFC fight data be used to build a probabilistic model for fight outcomes, and can those probabilities identify betting opportunities that the market has mispriced?

The project is not only about predicting winners. The main goal is to test whether a model can produce probabilities that are useful relative to sportsbook-implied probabilities.

## Important Methodology: 2010 Modeling Cutoff

I found a structural issue in the historical source data. UFC fights before 2010 showed implausibly high red-corner win rates, suggesting that the older rows may not preserve a consistent red/blue corner convention.

This matters because the target variable depends on corner assignment. If red/blue labeling is inconsistent over time, then the meaning of the target is not stable.

Because of this, I treat `2010-01-01` as the start of the reliable modeling era.

The full historical dataset is still retained for EDA and reference, but all predictive modeling uses only fights on or after `2010-01-01`, including:

- feature engineering for model inputs
- train/validation/test splits
- Logistic Regression, Random Forest, and XGBoost benchmarks
- odds matching
- betting backtests
- walk-forward market research

## Project Process

This project moved through six main stages:

1. **Data cleaning and leakage prevention**
   - Standardized fighter and fight-level data
   - Built leakage-safe pre-fight cumulative features
   - Created recency, Elo, style, cardio, and matchup features
   - Excluded current-fight and post-fight information from model inputs
   - Restricted predictive modeling to the post-2010 reliable modeling universe

2. **Validation-first model development**
   - Used chronological train/validation/test splits
   - Used validation data for feature evaluation and model selection
   - Started with interpretable Logistic Regression
   - Tested grouped hypotheses, ablations, recency features, and feature-selection ideas without using the test set

3. **Frozen final benchmark models**
   - Finalized a lean common feature set
   - Evaluated frozen Logistic Regression, Random Forest, and XGBoost once on the hold-out test set
   - After the post-2010 cutoff, Logistic Regression had the best test log loss, ROC AUC, and Brier score
   - Random Forest had the best test accuracy
   - XGBoost did not win the final comparison

4. **Odds ingestion and merge**
   - Cleaned the Kaggle UFC odds dataset
   - Selected a single odds source instead of averaging across books
   - Converted odds to no-vig implied probabilities
   - Merged odds into the post-2010 modeling dataset

5. **Exploratory betting backtest**
   - Used frozen model probabilities and no-vig market probabilities
   - Compared model probability to market-implied probability to estimate edge
   - Simulated flat-bet and fractional Kelly staking
   - Found that the first betting backtest was negative across tested thresholds
   - Treated threshold comparisons as descriptive diagnostics, not model tuning

6. **Walk-forward market research**
   - Built a separate pre-test walk-forward workflow on the odds-matched train/validation era
   - Used annual expanding folds with a calibration year and next-year test slice
   - Compared raw model probabilities, calibrated model probabilities, no-vig market probabilities, and model-market blends
   - Found that calibration improved model probabilities somewhat, but the market remained the strongest standalone probability benchmark


## Main Conclusion

UFC fight outcomes are meaningfully predictable, but profitable betting is much harder than pure classification.

The model learned real signal from historical fight data, but the betting and walk-forward results suggest that much of this signal is already reflected in market odds. The most important takeaway is that predictive performance does not automatically imply betting edge.

The strongest future direction is not simply adding more models, but improving probability calibration, using cleaner market benchmarks, and testing whether any model signal remains after conditioning on market-implied probabilities.

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

final_comparison = pd.read_csv(repo_root / "outputs" / "final_model_test_comparison.csv")
walk_forward_comparison = pd.read_csv(repo_root / "outputs" / "walk_forward_model_comparison.csv")

print("Final frozen hold-out model comparison:")
display(final_comparison)

print("Pre-test walk-forward market benchmark comparison:")
display(walk_forward_comparison)
